In [ ]:
import json
import networkx as nx
import re
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def glossary_to_edgelist(json_data):
    """Converts a glossary JSON data to an edgelist (source/target pairs)."""
    edgelist = []
    for edge_data in json_data.get("edges", []):
        source_id = edge_data.get('id')
        definition = edge_data.get('definition', '')
        defin = word_tokenize(definition)
        defin_sw = [word for word in defin if word.lower() not in stopwords.words()] #lower case added.

        # Join the filtered words back into a string for regex.
        filtered_definition = " ".join(defin_sw)
        
        targets = re.findall(r'\b\w+\b', filtered_definition)
        for target in targets:
            edgelist.append((source_id, target))
    return edgelist

def build_graph_from_edgelist(edgelist):
    """Builds a NetworkX graph from an edgelist."""
    G = nx.DiGraph()
    G.add_edges_from(edgelist)
    return G

# File Path: Replace with the actual path to your JSON file.
json_file = 'percyjackson.json'

# Load the JSON data from the file.
with open(json_file, 'r') as f:
    json_data = json.load(f)

edgelist = glossary_to_edgelist(json_data)
print(edgelist)
G = build_graph_from_edgelist(edgelist)
pos = nx.spring_layout(G)  # Layout algorithm
nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray')
plt.show()

# Extract source-target pairs.
source_target_pairs = list(G.edges())

# Export to JSON using pandas with orient='records'.
df = pd.DataFrame(source_target_pairs, columns=['source', 'target'])
df.to_json('source_target_graph.json', orient='records', indent=4) #indent makes it pretty.

print("Source-target graph exported to source_target_graph.json")


# Optional: Print the first few rows of the DataFrame.
print(df.head())

In [ ]:
import json

def json_to_adjacency_matrix(json_data):
    try:
        graph_data = json.loads(json_data)

        if "edges" in graph_data:
            edges = graph_data["edges"]
            unique_nodes = set()
            for edge in edges:
                unique_nodes.add(edge["source"])
                unique_nodes.add(edge["target"])

            node_id_map = {node: index for index, node in enumerate(unique_nodes)}
            num_nodes = len(unique_nodes)

            adjacency_matrix = [[0] * num_nodes for _ in range(num_nodes)]

            for edge in edges:
                source_id = node_id_map[edge["source"]]
                target_id = node_id_map[edge["target"]]
                if 0 <= source_id < num_nodes and 0 <= target_id < num_nodes:
                    adjacency_matrix[source_id][target_id] = 1
                else:
                    print(f"Invalid node ID: {edge['source']}, {edge['target']}")

            return [node_id_map, adjacency_matrix]

        elif "nodes" in graph_data:
            nodes = graph_data["nodes"]
            num_nodes = len(nodes)

            adjacency_matrix = [[0] * num_nodes for _ in range(num_nodes)]

            for node in nodes:
                node_id = node["id"]
                if "neighbors" in node:
                    for neighbor_id in node["neighbors"]:
                        if 0 <= node_id < num_nodes and 0 <= neighbor_id < num_nodes:
                            adjacency_matrix[node_id][neighbor_id] = 1
                        else:
                            print(f"Invalid node ID or neighbor ID: {node_id}, {neighbor_id}")

            return adjacency_matrix

        else:
            print("JSON data must contain 'edges' or 'nodes' array.")
            return None

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return None

def find_min_degree_node(graph):
    min_degree = float('inf')
    min_node = -1

    for i, row in enumerate(graph):
        degree = sum(row)
        if degree < min_degree and degree >= 3:
            min_degree = degree
            min_node = i
    return min_node

def create_containers(graph, start_node):
    num_nodes = len(graph)
    distances = [float('inf')] * num_nodes
    containers = {}
    queue = [(start_node, 0)]
    edges = []

    distances[start_node] = 0

    while queue:
        node, distance = queue.pop(0)

        if distance not in containers:
            containers[distance] = []
        containers[distance].append(node)

        for neighbor, connected in enumerate(graph[node]):
            if connected == 1 and distances[neighbor] == float('inf'):
                distances[neighbor] = distance + 1
                queue.append((neighbor, distance + 1))

    container_list = list(containers.values())

    for i, current_container in enumerate(container_list):
        for source_node in current_container:
            for j in range(1, i + 1):
                source_index = current_container.index(source_node)
                target_index = (source_index + j) % len(current_container)
                target_node = current_container[target_index]
                # Check if edge already exists
                if {"source": source_node, "target": target_node} not in edges:
                    edges.append({"source": source_node, "target": target_node})

        if i < len(container_list) - 1:
            next_container = container_list[i + 1]
            for source_node in current_container:
                for target_node in next_container:
                    edges.append({"source": source_node, "target": target_node})

    return {"containers": container_list, "edges": edges}

def process_graph_from_file(filepath):
    try:
        with open(filepath, 'r') as file:
            json_data = file.read()

        node_map, adjacency_matrix = json_to_adjacency_matrix(json_data)
        node_map2 = {y: x for x, y in node_map.items()}
        
#        print(node_map2)

        if adjacency_matrix:
            start_node = find_min_degree_node(adjacency_matrix)
            result = create_containers(adjacency_matrix, start_node)
            print("Containers (Indices):", result["containers"])
            print("Edges:", result["edges"])

            # Translate indices to terms.
    
            translated_containers = translate_indices_to_terms(result["containers"], node_map2)
            print("Containers (Terms):", translated_containers)

        else:
            print("Failed to create adjacency matrix.")

    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")
def process_graph(json_data):
    node_map, adjacency_matrix = json_to_adjacency_matrix(json_data)
    node_map_reverse = {y: x for x, y in node_map.items()}

    if adjacency_matrix is not None:
        start_node_index = find_min_degree_node(adjacency_matrix)
        graph_info = create_containers(adjacency_matrix, start_node_index)
        containers_indices = graph_info["containers"]
        edges = graph_info["edges"]
        containers_terms = translate_indices_to_terms(containers_indices, node_map_reverse)

        return {
            "containers_indices": containers_indices,
            "containers_terms": containers_terms,
            "edges": edges,
            "node_map": node_map
        }
    else:
        return {
            "containers_indices": [],
            "containers_terms": [],
            "edges": [],
            "node_map": {}
        }

def translate_indices_to_terms(indices_lists, node_id_to_term):
    """Translates node indices to terms using the provided mapping."""

    print(node_id_to_term[indices_lists[0][0]], node_id_to_term[indices_lists[1][0]])
    print(node_id_to_term[0])
    translated_lists = []
    for indices in indices_lists:
        translated_terms = [node_id_to_term.get(index, f"Unknown ID: {index}") for index in indices] #Removed the str()
        translated_lists.append(translated_terms)
    return translated_lists

def load_glossary_and_create_mapping(json_file):
    """Loads glossary and creates a node ID to term mapping."""
    with open(json_file, 'r') as f:
        glossary = json.load(f)

    node_id_to_term = {}
    for edge_data in glossary.get("edges", []):
        node_id_to_term[edge_data.get('id')] = edge_data.get('id') #for this case, id is the term.
    return node_id_to_term

# Example Usage:
filepath = 'source_target_graph.json'  # Replace with your file path
process_graph_from_file(filepath)


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def visualize_graph(containers, edges):
    G = nx.DiGraph()

    # Add nodes
    nodes = set()
    for container in containers:
        nodes.update(container)
    G.add_nodes_from(nodes)

    # Add edges
    for edge in edges:
        print(edge)
        G.add_edge(edge['source'], edge['target'])

    # Draw the graph
    pos = nx.spring_layout(G)  # Layout algorithm
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray')
    plt.show()


# Example usage (using the output from your previous example):
containers = [[0], [1, 2, 3, 4, 5, 6], [7, 8, 9, 10, 11], [12, 13, 14, 15]]
edges = [{'source': 0, 'target': 1}, {'source': 0, 'target': 2}, {'source': 0, 'target': 3}, 
         {'source': 0, 'target': 4}, {'source': 0, 'target': 5}, {'source': 0, 'target': 6}, 
         {'source': 1, 'target': 2}, {'source': 2, 'target': 3}, {'source': 3, 'target': 4}, 
         {'source': 4, 'target': 5}, {'source': 5, 'target': 6}, {'source': 6, 'target': 1}, 
         {'source': 1, 'target': 7}, {'source': 1, 'target': 8}, {'source': 1, 'target': 9}, 
         {'source': 1, 'target': 10}, {'source': 1, 'target': 11}, {'source': 2, 'target': 7}, 
         {'source': 2, 'target': 8}, {'source': 2, 'target': 9}, {'source': 2, 'target': 10}, 
         {'source': 2, 'target': 11}, {'source': 3, 'target': 7}, {'source': 3, 'target': 8}, 
         {'source': 3, 'target': 9}, {'source': 3, 'target': 10}, {'source': 3, 'target': 11}, 
         {'source': 4, 'target': 7}, {'source': 4, 'target': 8}, {'source': 4, 'target': 9}, 
         {'source': 4, 'target': 10}, {'source': 4, 'target': 11}, {'source': 5, 'target': 7}, 
         {'source': 5, 'target': 8}, {'source': 5, 'target': 9}, {'source': 5, 'target': 10}, 
         {'source': 5, 'target': 11}, {'source': 6, 'target': 7}, {'source': 6, 'target': 8}, 
         {'source': 6, 'target': 9}, {'source': 6, 'target': 10}, {'source': 6, 'target': 11}, 
         {'source': 7, 'target': 8}, {'source': 7, 'target': 9}, {'source': 8, 'target': 9}, 
         {'source': 8, 'target': 10}, {'source': 9, 'target': 10}, {'source': 9, 'target': 11}, 
         {'source': 10, 'target': 11}, {'source': 10, 'target': 7}, {'source': 11, 'target': 7}, 
         {'source': 11, 'target': 8}, {'source': 7, 'target': 12}, {'source': 7, 'target': 13}, 
         {'source': 7, 'target': 14}, {'source': 7, 'target': 15}, {'source': 8, 'target': 12}, 
         {'source': 8, 'target': 13}, {'source': 8, 'target': 14}, {'source': 8, 'target': 15}, 
         {'source': 9, 'target': 12}, {'source': 9, 'target': 13}, {'source': 9, 'target': 14}, 
         {'source': 9, 'target': 15}, {'source': 10, 'target': 12}, {'source': 10, 'target': 13}, 
         {'source': 10, 'target': 14}, {'source': 10, 'target': 15}, {'source': 11, 'target': 12}, 
         {'source': 11, 'target': 13}, {'source': 11, 'target': 14}, {'source': 11, 'target': 15}, 
         {'source': 12, 'target': 13}, {'source': 12, 'target': 14}, {'source': 12, 'target': 15}, 
         {'source': 13, 'target': 14}, {'source': 13, 'target': 15}, {'source': 13, 'target': 12}, 
         {'source': 14, 'target': 15}, {'source': 14, 'target': 12}, {'source': 14, 'target': 13}, 
         {'source': 15, 'target': 12}, {'source': 15, 'target': 13}, {'source': 15, 'target': 14}]

#visualize_graph(containers, edges)

In [ ]:
    import nltk
    nltk.download('popular')  # or specific packages like 'punkt', 'wordnet'

In [ ]:
def generate_algorithmic_graph_v2(min_degree):
    """
    Generates a directed graph representation with neighborhood information.

    Args:
        min_degree (int): The minimum out-degree of the starting node (node 0).

    Returns:
        dict: A dictionary representing the graph where keys are node IDs
              and values are dictionaries containing 'targets' and 'neighborhood'.
    """
    graph_representation = {}
    num_container1 = min_degree
    num_container2 = 5  # Example size for the next container
    num_container3 = 4  # Example size for the container after that

    num_nodes = 1 + num_container1 + num_container2 + num_container3

    containers = [
        [0],
        list(range(1, 1 + num_container1)),
        list(range(1 + num_container1, 1 + num_container1 + num_container2)),
        list(range(1 + num_container1 + num_container2, num_nodes)),
    ]

    # Initialize graph representation
    for i in range(num_nodes):
        graph_representation[str(i)] = {"targets": [], "neighborhood": ""}

    # Assign neighborhoods
    for i, container in enumerate(containers):
        neighborhood_label = f"C{i}"
        for node in container:
            graph_representation[str(node)]["neighborhood"] = neighborhood_label

    # Edges from the minimum degree node (container 0) to container 1
    for i in range(1, 1 + num_container1):
        graph_representation["0"]["targets"].append(i)

    # Circular edges within container 1
    for i in range(1, num_container1 + 1):
        target = (i % num_container1) + 1
        graph_representation[str(i)]["targets"].append(target)

    # Edges from container 1 to container 2 (fully connected)
    for source in containers[1]:
        for target in containers[2]:
            graph_representation[str(source)]["targets"].append(target)

    # Circular edges within container 2
    d = 1 + min_degree  # Starting index of container 2
    for i in range(num_container1 + 1, num_container1 + 1 + num_container2):
        target_offset = (i - d + 1) % num_container2 + d
        graph_representation[str(i)]["targets"].append(target_offset)

    # Edges from container 2 to container 3 (fully connected)
    d2 = 1 + min_degree + num_container2 # Starting index of container 3
    for source in containers[2]:
        for target in containers[3]:
            graph_representation[str(source)]["targets"].append(target)

    # Circular edges within container 3
    d3 = 1 + min_degree + num_container2 # Starting index of container 3
    num_c3 = len(containers[3])
    for i in range(d2, num_nodes):
        target_offset = ((i - d3 + 1) % num_c3) + d3
        graph_representation[str(i)]["targets"].append(target_offset)

    return graph_representation

# Example usage: Generate a graph with minimum degree 6
min_degree_node = 0
algorithmic_graph_v2 = generate_algorithmic_graph_v2(min_degree_node)
print(algorithmic_graph_v2)

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer  # Or use WordNetLemmatizer
from nltk.corpus import stopwords

# Download NLTK resources (if not already downloaded)
nltk.download('punkt')
nltk.download('stopwords')

def normalize_text(text):
    tokens = word_tokenize(text.lower())  # Lowercase and tokenize
    # Remove punctuation (example)
    tokens = [word for word in tokens if word.isalnum()]
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Stemming (example)
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return tokens

def analyze_graph_with_normalized_tokens(graph, containers, edges, node_map):
    normalized_nodes = {}
    for node_index, node_term in node_map.items():
        normalized_nodes[node_index] = normalize_text(node_term)

    # ... (rest of your analysis code, using normalized_nodes)
    # Example output.
    print("Normalized Nodes:", normalized_nodes)



In [ ]:
def generate_algorithmic_graph(min_degree):
    """
    Generates a directed graph with a specified minimum out-degree for the starting node.

    Args:
        min_degree (int): The minimum out-degree of the starting node (node 0).

    Returns:
        tuple: A tuple containing the containers (list of lists) and edges (list of dictionaries).
    """

    nun_containers = min_degree - 1;
    
    contaiers = [];
    for i in range (0, num_containers):
        if (i == 0):
            containers.append([0]);
        else
            containers.append([]);

    for i in range(0, num_containers):
        for j in range(min_degree - i):
            containers[i].append([j]);
                
#    ]
    edges = []

    # Edges from the minimum degree node (container 0) to container 1
    for i in range(1, 1 + num_container1):
        edges.append({"source": 0, "target": i})

    # Circular edges within container 1
    for i in range(1, num_container1 + 1):
        target = (i % num_container1) + 1
        edges.append({"source": i, "target": target})

    # Edges from container 1 to container 2 (fully connected)
    for source in containers[1]:
        for target in containers[2]:
            edges.append({"source": source, "target": target})

    # Circular edges within container 2
    for i in range(num_container1 + 1, num_container1 + 1 + num_container2):
        target = ((i - (num_container1 + 1) + 1) % num_container2) + (num_container1 + 1)
        edges.append({"source": i, "target": target})

    # Edges from container 2 to container 3 (fully connected)
    for source in containers[2]:
        for target in containers[3]:
            edges.append({"source": source, "target": target})

    # Circular edges within container 3
    for i in range(num_container1 + num_container2 + 1, num_nodes):
        target = ((i - (num_container1 + num_container2 + 1) + 1) % num_container3) + (num_container1 + num_container2 + 1)
        edges.append({"source": i, "target": target})

    return {"Containers": containers, "Edges": edges}

# Example usage: Generate a graph with minimum degree 6
min_degree_node = 10
algorithmic_graph = generate_algorithmic_graph(min_degree_node)
print("Containers:", algorithmic_graph["Containers"])
print("Edges:", algorithmic_graph["Edges"])

In [ ]:
import time
import networkx as nx

def analyze_algorithm_report(graph, containers, edges, node_map):
    G = nx.DiGraph()
    for edge in edges:
        G.add_edge(edge["source"], edge["target"])

    # 1. Minimum Degree
    min_degree_node_index = find_min_degree_node(graph)
    min_degree = min(G.out_degree(node) for node in G.nodes())

    # 2. Number of Nodes
    num_nodes = len(G.nodes())

    # 3. Number of Edges
    num_edges = len(edges)

    # 4. Number of Containers
    num_containers = len(containers)

    # 5. Seymour Nodes (exterior vs. interior, considering predecessors)
    degree_doubling_nodes = {}
    for i in range(len(containers)):
        for node in containers[i]:
            interior_degree = 0
            exterior_degree = 0

            # Check successors (outgoing edges)
            for successor in G.successors(node):
                if successor in containers[i]:
                    interior_degree += 1
                elif successor not in containers[i] and (i == 0 or successor not in containers[i - 1]):
                    exterior_degree += 1

            # Check predecessors (incoming edges)
            for predecessor in G.predecessors(node):
                if predecessor in containers[i]:
                    interior_degree += 1  # Count incoming as interior
                elif predecessor not in containers[i] and (i == 0 or predecessor not in containers[i - 1]):
                    exterior_degree += 1 # Count incoming as exterior

            print(node, " - ", interior_degree, exterior_degree)
            if exterior_degree > interior_degree:
                degree_doubling_nodes[node_map[node]] = 1
        print("seymour_nodes = ", seymour_nodes)

    # 6. Run Time
    start_time = time.time()
    # (Your graph analysis functions here)
    end_time = time.time()
    run_time = end_time - start_time

    # Output report
    print("Minimum Degree:", min_degree)
    print("Number of Nodes:", num_nodes)
    print("Number of Edges:", num_edges)
    print("Number of Containers:", num_containers)
    print("Seymour Nodes:", seymour_nodes)
    print("Run Time:", run_time, "seconds")